# Assignment 2: Grade Distribution using MapReduce



### Aim
To apply the MapReduce programming model to calculate the number of students falling under each grade category (S, A, B, C, D, E, F) for every subject, using previous semester final exam results as sample data.

### Description
MapReduce is a programming model used to process large datasets in a distributed manner by breaking the computation into three phases: Map, Shuffle, and Reduce. In this experiment, previous semester exam result data (StudentID, Subject, Grade) is used as sample input. In the Map phase, each result record is converted into a key-value pair of the form ((Subject, Grade), 1). In the Shuffle phase, these pairs are grouped together based on their key. Finally, in the Reduce phase, the values under each key are summed up to get the total number of students who received a particular grade in a particular subject. This experiment demonstrates how the MapReduce model can be used to efficiently perform aggregation tasks such as grade distribution analysis across multiple subjects.


In [1]:
# importing required libraries
import random
import pandas as pd
from collections import defaultdict

c:\Users\Dhiman\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:23: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
# sample previous semester result data - one record per student per subject
random.seed(42)

subjects = ["Data Warehousing and Big Data", "Software Engineering", "Computer Networks"]
grades = ["S", "A", "B", "C", "D", "E", "F"]
grade_weights = [10, 20, 25, 20, 15, 7, 3]   # more students land in average grades, fewer at extremes

exam_results = []
for student_id in range(2201, 2241):        # 40 students
    for subject in subjects:
        grade = random.choices(grades, weights=grade_weights, k=1)[0]
        exam_results.append({"StudentID": student_id, "Subject": subject, "Grade": grade})

print(f"Total records: {len(exam_results)}")
pd.DataFrame(exam_results).head(10)

Total records: 120


,StudentID,Subject,Grade
0,2201,Data Warehousing and Big Data,C
1,2201,Software Engineering,S
2,2201,Computer Networks,A
3,2202,Data Warehousing and Big Data,A
4,2202,Software Engineering,C
5,2202,Computer Networks,C
6,2203,Data Warehousing and Big Data,D
7,2203,Software Engineering,S
8,2203,Computer Networks,B
9,2204,Data Warehousing and Big Data,S


In [3]:
# map step - convert each result record into a (subject, grade) key with value 1
def mapper(record):
    key = (record["Subject"], record["Grade"])
    return key, 1

mapped = [mapper(record) for record in exam_results]
mapped[:5]

[(('Data Warehousing and Big Data', 'C'), 1),
 (('Software Engineering', 'S'), 1),
 (('Computer Networks', 'A'), 1),
 (('Data Warehousing and Big Data', 'A'), 1),
 (('Software Engineering', 'C'), 1)]

In [4]:
# shuffle step - group all the 1s together under their (subject, grade) key
def shuffle(pairs):
    grouped = defaultdict(list)
    for key, value in pairs:
        grouped[key].append(value)
    return grouped

grouped_data = shuffle(mapped)

In [5]:
# reduce step - sum up the grouped values to get the final count per key
def reducer(grouped_data):
    return {key: sum(values) for key, values in grouped_data.items()}

grade_counts = reducer(grouped_data)
grade_counts

{('Data Warehousing and Big Data', 'C'): 10,
 ('Software Engineering', 'S'): 7,
 ('Computer Networks', 'A'): 8,
 ('Data Warehousing and Big Data', 'A'): 10,
 ('Software Engineering', 'C'): 9,
 ('Computer Networks', 'C'): 9,
 ('Data Warehousing and Big Data', 'D'): 6,
 ('Computer Networks', 'B'): 9,
 ('Data Warehousing and Big Data', 'S'): 3,
 ('Software Engineering', 'A'): 11,
 ('Data Warehousing and Big Data', 'B'): 6,
 ('Computer Networks', 'D'): 6,
 ('Software Engineering', 'B'): 8,
 ('Data Warehousing and Big Data', 'E'): 2,
 ('Computer Networks', 'S'): 4,
 ('Software Engineering', 'D'): 3,
 ('Data Warehousing and Big Data', 'F'): 3,
 ('Computer Networks', 'E'): 2,
 ('Software Engineering', 'E'): 2,
 ('Computer Networks', 'F'): 2}

In [6]:
GRADE_ORDER = ["S", "A", "B", "C", "D", "E", "F"]

for subject in subjects:
    print(f"\nSubject: {subject}")
    print("Grade  Count")
    for grade in GRADE_ORDER:
        count = grade_counts.get((subject, grade), 0)
        print(f"  {grade}     {count}")


Subject: Data Warehousing and Big Data
Grade  Count
  S     3
  A     10
  B     6
  C     10
  D     6
  E     2
  F     3

Subject: Software Engineering
Grade  Count
  S     7
  A     11
  B     8
  C     9
  D     3
  E     2
  F     0

Subject: Computer Networks
Grade  Count
  S     4
  A     8
  B     9
  C     9
  D     6
  E     2
  F     2


In [7]:
# building a subject x grade summary table for a cleaner view
summary_rows = []
for subject in subjects:
    row = {"Subject": subject}
    for grade in GRADE_ORDER:
        row[grade] = grade_counts.get((subject, grade), 0)
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df

,Subject,S,A,B,C,D,E,F
0,Data Warehousing and Big Data,3,10,6,10,6,2,3
1,Software Engineering,7,11,8,9,3,2,0
2,Computer Networks,4,8,9,9,6,2,2
